# Lab 5: Function Approximation with Value Methods 

## Lab Assignment

Solve Gymnasium’s MountainCar-v0 using semi-gradient SARSA with tile coding. The observation space is Box([-1.2, -0.07], [0.6, 0.07]), representing position and velocity. Implement a tile coding feature constructor from scratch using NumPy with configurable number of tilings, tiles per dimension, and offset patterns. Start with 8 tilings of 8×8 tiles and experiment with variations.

Your tile coding implementation should: (1) discretize the continuous observation space into overlapping grids, (2) for each tiling, compute which tile the state falls into, (3) return a sparse binary feature vector (or use a list of active tile indices for efficiency). For each action, maintain a separate weight vector as a NumPy array.

Your RL implementation should: (1) extract the continuous observation from env.step(), (2) convert it to a feature vector using tile coding, (3) compute Q(s,a) as a linear function: Q = np.dot(weights[a], features), (4) perform semi-gradient updates after each step: weights[a] += alpha * (target - Q) * features. Implement ε-greedy action selection by computing Q-values for all actions.

Track episodes-to-goal over learning. Create visualizations of: (1) the learned value function as a heatmap over the 2D state space (position × velocity) by evaluating max_a Q(s,a) on a grid using NumPy’s meshgrid, (2) convergence curves for different feature configurations, (3) the learned policy shown as action choices across the state space, (4) sample trajectories from various starting positions overlaid on the value function.

Document how feature design choices (number of tilings, tile sizes, offsets) affect learning speed, final performance, and computational cost. Explain why MountainCar is difficult for tabular methods (hint: continuous state space would require infinite states) and how function approximation enables solving it.

In [12]:
import sys
sys.path.insert(0, '..')

from src.environment import MountainCarEnvironment

# DONE: Environment Setup
# - Verify the environment loads and runs correctly
# - Understand the observation and action spaces before building anything

env = MountainCarEnvironment()

print("Observation space:", env.observation_space)
print("  Low: ", env.observation_space.low)
print("  High:", env.observation_space.high)
print()
print("Action space:", env.action_space)
print("  n actions:", env.action_space.n)
print()

obs, info = env.reset()
print("Initial observation:", obs)

obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print("After one step:")
print("  obs:", obs)
print("  reward:", reward)
print("  terminated:", terminated, " truncated:", truncated)

Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
  Low:  [-1.2  -0.07]
  High: [0.6  0.07]

Action space: Discrete(3)
  n actions: 3

Initial observation: [-0.4452088  0.       ]
After one step:
  obs: [-0.44679132 -0.00158252]
  reward: -1.0
  terminated: False  truncated: False


In [13]:
from src.tile_coder import TileCoder

# DONE: Tile Coding Feature Constructor
# - Build a tile coding component that maps a continuous observation to a feature representation
# - Support configurable number of tilings, tiles per dimension, and offset patterns
# - Verify correctness: the same state should always produce the same features

tc = TileCoder()

print("n_tilings:    ", tc.n_tilings)
print("tiles_per_dim:", tc.tiles_per_dim)
print("state_bounds: ", tc.state_bounds)
print("n_dims:       ", tc.n_dims)
print("tile_widths:  ", tc.tile_widths)
print("num_features: ", tc.num_features)
print()

# Verify: one active index per tiling
mid_state = [-0.5, 0.0]
tiles = tc.get_tiles(mid_state)
print(f"Active tiles for state {mid_state}: {tiles}")
print(f"Count: {len(tiles)} (expected {tc.n_tilings})")
print()

# Verify: all indices within valid range
assert all(0 <= i < tc.num_features for i in tiles), "Index out of range!"
print("All indices within [0, num_features): OK")
print()

# Verify: deterministic
assert tc.get_tiles(mid_state) == tc.get_tiles(mid_state), "Not deterministic!"
print("Determinism: OK")
print()

# Verify: different states produce different tiles
other_state = [0.0, 0.03]
assert tc.get_tiles(mid_state) != tc.get_tiles(other_state), "Different states should differ!"
print(f"Active tiles for state {other_state}: {tc.get_tiles(other_state)}")
print("Different states produce different tiles: OK")
print()

# Verify: boundary states don't crash
for label, state in [("lower bound", [-1.2, -0.07]), ("upper bound", [0.6, 0.07])]:
    t = tc.get_tiles(state)
    assert len(t) == tc.n_tilings
    print(f"{label} {state}: {t}")

n_tilings:     8
tiles_per_dim: [8, 8]
state_bounds:  [(-1.2, 0.6), (-0.07, 0.07)]
n_dims:        2
tile_widths:   [0.225  0.0175]
num_features:  512

Active tiles for state [-0.5, 0.0]: [28, 92, 156, 220, 284, 348, 412, 476]
Count: 8 (expected 8)

All indices within [0, num_features): OK

Determinism: OK

Active tiles for state [0.0, 0.03]: [45, 109, 173, 238, 302, 366, 438, 502]
Different states produce different tiles: OK

lower bound [-1.2, -0.07]: [0, 64, 128, 192, 256, 320, 384, 448]
upper bound [0.6, 0.07]: [63, 127, 191, 255, 319, 383, 447, 511]


In [3]:
# TODO: Semi-Gradient SARSA Agent
# - Implement action-value estimation as a linear function over tile-coded features
# - Implement epsilon-greedy action selection
# - Implement the semi-gradient SARSA weight update rule

In [4]:
# TODO: Training
# - Train the agent on MountainCar-v0 using a baseline configuration
# - Track episode length over time as the primary performance signal
# - Confirm the agent is learning (episode lengths should decrease)

In [5]:
# TODO: Visualization: Value Function
# - Show the learned value function as a heatmap over the 2D state space (position x velocity)

In [6]:
# TODO: Visualization: Learned Policy
# - Show the greedy action chosen by the agent across the 2D state space

In [7]:
# TODO: Visualization: Sample Trajectories
# - Show trajectories from several starting positions overlaid on the value function heatmap

In [8]:
# TODO: Feature Design Experiments
# - Compare the effect of varying number of tilings and tile resolution on learning
# - Measure impact on learning speed, final performance, and computational cost
# - Results feed directly into the convergence curves visualization and the analysis

In [9]:
# TODO: Visualization: Convergence Curves
# - Show how episode length evolves over training
# - Required: curves for multiple feature configurations on the same plot (not just the baseline)

In [10]:
# TODO: Analysis
# - Explain why MountainCar is difficult for tabular methods
# - Explain how tile coding addresses this and what trade-offs it introduces
# - Connect the experimental results to the theory